# 02 - Exploratory Data Analysis (Objective 2)
Understand structure, distributions, relationships and text characteristics **before** modelling.

In [ ]:
%pip install -q pandas seaborn matplotlib pyarrow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style="whitegrid")
df = pd.read_parquet("data/raw/reviews_raw.parquet")
df["review_date"] = pd.to_datetime(df["review_date"], errors="coerce")
print(df.shape)
df.head()

## 1. Structure and data quality

In [ ]:
df.info()
print("\nMissing values (%):")
print((df.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate review_id:", df["review_id"].duplicated().sum())

## 2. Ratings and review volume

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(data=df, x="rating", ax=ax[0])
ax[0].set_title("Rating distribution (note any invalid values)")

monthly = df.dropna(subset=["review_date"]).set_index("review_date").resample("ME")["review_id"].count()
monthly.plot(ax=ax[1], marker="o")
ax[1].set_title("Reviews per month")
plt.tight_layout(); plt.show()

## 3. Text characteristics

In [ ]:
df["review_text"] = df["review_text"].fillna("")
df["n_words"] = df["review_text"].str.split().str.len()
df["n_chars"] = df["review_text"].str.len()

print(df[["n_words", "n_chars"]].describe().round(1))
print("\nEmpty reviews:", (df["n_words"] == 0).sum())
print("Very short reviews (<4 words):", (df["n_words"] < 4).sum())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["n_words"], bins=40, ax=ax[0]); ax[0].set_title("Review length (words)")
sns.boxplot(data=df, x="rating", y="n_words", ax=ax[1]); ax[1].set_title("Length by rating")
plt.tight_layout(); plt.show()

## 4. Hotel, guest and booking patterns

In [ ]:
clean_rating = df[df["rating"].between(1, 5)]

fig, ax = plt.subplots(2, 2, figsize=(14, 9))
clean_rating.groupby("hotel_name")["rating"].mean().sort_values().plot.barh(ax=ax[0, 0], title="Average rating by hotel")
clean_rating.groupby("booking_channel")["rating"].mean().plot.bar(ax=ax[0, 1], title="Average rating by booking channel")
clean_rating.groupby("guest_type")["rating"].mean().plot.bar(ax=ax[1, 0], title="Average rating by guest type")
clean_rating.groupby("guest_country")["rating"].count().plot.bar(ax=ax[1, 1], title="Reviews by guest country")
plt.tight_layout(); plt.show()

df["length_of_stay"] = (pd.to_datetime(df["check_out"]) - pd.to_datetime(df["check_in"])).dt.days
print(df[["rating", "length_of_stay", "total_price", "n_words"]].corr(numeric_only=True).round(2))

## 5. Rating vs. review content (most common words)

In [ ]:
STOP = set("the a an and or but of to in on at for with was were is are it we i our my this that as be had have very".split())

def top_words(texts, n=12):
    words = " ".join(texts).lower().replace(".", " ").replace(",", " ").split()
    return Counter(w for w in words if w not in STOP and len(w) > 2).most_common(n)

print("1-2 star reviews:", top_words(df.loc[df["rating"].isin([1, 2]), "review_text"]))
print("\n4-5 star reviews:", top_words(df.loc[df["rating"].isin([4, 5]), "review_text"]))

In [ ]:
# Reviews where rating and text may disagree (important for labelling quality!)
NEG = ["rude", "dirty", "filthy", "terrible", "awful", "broken", "poor"]
POS = ["excellent", "friendly", "spotless", "perfect", "fantastic", "delicious"]

def has_any(t, words): return any(w in t.lower() for w in words)

suspect_high = df[(df["rating"] >= 4) & df["review_text"].apply(lambda t: has_any(t, NEG))]
suspect_low = df[(df["rating"] <= 2) & df["review_text"].apply(lambda t: has_any(t, POS))]
print("High rating but negative words:", len(suspect_high))
print("Low rating but positive words:", len(suspect_low))
suspect_high[["rating", "review_text"]].head(3)

### Findings to write in your EDA report
- Data quality problems found (missing, duplicates, invalid ratings, future dates).
- Rating imbalance (this drives the class-weight decision later).
- Review length distribution (this drives the BERT `max_length` decision).
- Hotels/segments with unusual ratings.
- Share of rating/text disagreement.

Optional quick profile: `pip install ydata-profiling`, then `ProfileReport(df).to_file("reports/eda_profile.html")`.